In [19]:
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import preprocessing as pps
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV, KFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import os

On importe notre data et on la netoie avec notre fonction data_clean_2

In [21]:
data = pd.read_csv(r"../data/dataset.csv")
data_clean_2=pps.clean_data_2(data)
print(data.shape)
print(data_clean_2.shape)

(768, 9)
(708, 11)


On recrée nos clusters avec notre fonction make_cluster

In [22]:
colonnes_cluster=['Glucose','BloodPressure','BMI','DiabetesPedigreeFunction','Age']

In [23]:
data_clean_2= pps.make_cluster(data_clean_2,2,colonnes_cluster,True)
data_clean_2.head()

,index_tab,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Insuline_bool,SkinThickness_bool,cluster
0,0,6,148,72,35,0,33.6,0.627,50,0,1,0
1,1,1,85,66,29,0,26.6,0.351,31,0,1,1
2,2,8,183,64,0,0,23.3,0.672,32,0,0,0
3,3,1,89,66,23,94,28.1,0.167,21,1,1,1
5,5,5,116,74,0,0,25.6,0.201,30,0,0,1


## Entrainement des modèles

Metriques utilisées:

Précision (Precision)
La précision est une métrique qui mesure la proportion d'identifications positives qui étaient réellement correctes. Elle est calculée en divisant le nombre de vrais positifs par la somme des vrais positifs et des faux positifs (VP / (VP + FP)). Une précision élevée indique que le modèle a un faible taux de faux positifs.

Rappel (Recall)
Le rappel, aussi appelé sensibilité, mesure la proportion de positifs réels qui ont été correctement identifiés par le modèle. Il est calculé en divisant le nombre de vrais positifs par la somme des vrais positifs et des faux négatifs (VP / (VP + FN)). Un rappel élevé signifie que le modèle a un faible taux de faux négatifs.

F1 score
Le F1-score (ou score F1) est une métrique d'évaluation des modèles de classification qui combine la précision (precision) et le rappel (recall) en une seule valeur. Il est particulièrement utile dans les cas où les classes sont déséquilibrées.

### Division des données

In [24]:
X=data_clean_2[colonnes_cluster]
y=data_clean_2['cluster']

X_train, X_test, y_train, y_test=train_test_split(X, y,test_size=0.20,random_state=42,stratify=y )
X_train.shape, y_train.shape

((566, 5), (566,))

In [25]:
#On récupere le nom de nos classe dans notre variable cible

class_names = sorted(y.unique().astype(str))
print("Classes dans la cible : ", class_names)

Classes dans la cible :  [np.str_('0'), np.str_('1')]


In [26]:
#Affiche la distribution des classes dans le training set

class_distribution = y_train.value_counts(normalize=True)
print("\nDistribution des classes dans le training set:")
for class_label, proportion in class_distribution.items():
    risk_level = "Faible risque" if class_label == 1 else "Haut risque"
    print(f"   {risk_level}: {proportion:.3f} ({y_train.value_counts()[class_label]} échantillons)")


Distribution des classes dans le training set:
   Faible risque: 0.562 (318 échantillons)
   Haut risque: 0.438 (248 échantillons)


On utilise RandomOverSampler car nous n'avons pas beaucoup de data et la classe minoritaire à haut rique est sous_représenté et nous ne voulons pas perdre d'information.
Principe :Crée artificiellement des copies d’exemples de la classe minoritaire pour atteindre la même taille que la classe majoritaire.

On utilise egalement le StandardScaler pour les modèles qui en on besoins

In [27]:
pipelines_and_params = {
    "Random Forest": (
        Pipeline([
            ('ros', RandomOverSampler(random_state=42)),
            ('clf', RandomForestClassifier(random_state=42))
        ]),
        {
            'clf__n_estimators': [100, 200, 300],
            'clf__max_depth': [None, 5, 10],
            'clf__min_samples_split': [2, 5, 10]
        }
    ),
    "Gradient Boosting": (
        Pipeline([
            ('ros', RandomOverSampler(random_state=42)),
            ('scaler', StandardScaler()),
            ('clf', GradientBoostingClassifier(random_state=42))
        ]),
        {
            'clf__n_estimators': [100, 200, 300],
            'clf__learning_rate': [0.01, 0.05, 0.1],
            'clf__max_depth': [3, 4, 5]
        }
    ),
    "SVM": (
        Pipeline([
            ('ros', RandomOverSampler(random_state=42)),
            ('scaler', StandardScaler()),
            ('clf', SVC(random_state=42))
        ]),
        {
            'clf__C': [0.1, 1, 10],
            'clf__kernel': ['rbf', 'linear']
        }
    ),
    "Régression Logistique": (
        Pipeline([
            ('ros', RandomOverSampler(random_state=42)),
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(max_iter=200, random_state=42))
        ]),
        {
            'clf__C': [0.01, 0.1, 1, 10],
            'clf__penalty': ['l2']
        }
    )
}

In [28]:
models_best_estimators = {}
results_summary = []

for name, (pipe, param_grid) in pipelines_and_params.items():
    search = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring='f1_weighted',
        cv=5,
        n_jobs=1,
        verbose=0,  
        refit=True
    )
    search.fit(X_train, y_train)
    best_estimator = search.best_estimator_

    modele_key = f"Modele_{name.replace(' ', '_')}"
    models_best_estimators[modele_key] = best_estimator

    y_pred = best_estimator.predict(X_test)

    best_score_cv = search.best_score_
    best_test_f1 = classification_report(y_test, y_pred, output_dict=True)["weighted avg"]["f1-score"]

    results_summary.append({
        "Modèle": name,
        "CV moyen_(f1_weighted)": best_score_cv,
        "F1 test": best_test_f1
    })

df_results = pd.DataFrame(results_summary)

print("\nRésumé comparatif des performances :\n")
print(df_results)



Résumé comparatif des performances :

                  Modèle  CV moyen_(f1_weighted)   F1 test
0          Random Forest                0.950440  0.971876
1      Gradient Boosting                0.945210  0.971831
2                    SVM                0.992951  0.985887
3  Régression Logistique                0.996491  0.985887


In [29]:
for model, valeur in models_best_estimators.items():
    print(model, valeur)
    print()

Modele_Random_Forest Pipeline(steps=[('ros', RandomOverSampler(random_state=42)),
                ('clf',
                 RandomForestClassifier(n_estimators=300, random_state=42))])

Modele_Gradient_Boosting Pipeline(steps=[('ros', RandomOverSampler(random_state=42)),
                ('scaler', StandardScaler()),
                ('clf',
                 GradientBoostingClassifier(learning_rate=0.05,
                                            n_estimators=300,
                                            random_state=42))])

Modele_SVM Pipeline(steps=[('ros', RandomOverSampler(random_state=42)),
                ('scaler', StandardScaler()),
                ('clf', SVC(C=10, kernel='linear', random_state=42))])

Modele_Régression_Logistique Pipeline(steps=[('ros', RandomOverSampler(random_state=42)),
                ('scaler', StandardScaler()),
                ('clf',
                 LogisticRegression(C=10, max_iter=200, random_state=42))])



In [30]:
for modele_name, pipeline in models_best_estimators.items():
    print(f"\n=== Évaluation du modèle : {modele_name} ===")
    
    # Cross-validation stratifiée F1-score (pondéré)
    cv_scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring='f1_weighted',
        n_jobs=1
    )
    print(f"Cross-validation F1-score (moyenne) : {cv_scores.mean():.3f}")
    print(f"Cross-validation F1-score (écart-type) : {cv_scores.std():.3f}")
    
    # Entraînement complet sur X_train
    pipeline.fit(X_train, y_train)
    
    # Prédiction sur X_test
    y_pred = pipeline.predict(X_test)
    
    # Matrice de confusion
    cm = confusion_matrix(y_test, y_pred)
    print("\nMatrice de confusion :")
    print(cm)
    
    # Rapport de classification détaillé
    cr = classification_report(y_test, y_pred, digits=3)
    print("\nRapport de classification :")
    print(cr)



=== Évaluation du modèle : Modele_Random_Forest ===
Cross-validation F1-score (moyenne) : 0.950
Cross-validation F1-score (écart-type) : 0.027

Matrice de confusion :
[[61  1]
 [ 3 77]]

Rapport de classification :
              precision    recall  f1-score   support

           0      0.953     0.984     0.968        62
           1      0.987     0.963     0.975        80

    accuracy                          0.972       142
   macro avg      0.970     0.973     0.971       142
weighted avg      0.972     0.972     0.972       142


=== Évaluation du modèle : Modele_Gradient_Boosting ===
Cross-validation F1-score (moyenne) : 0.945
Cross-validation F1-score (écart-type) : 0.022

Matrice de confusion :
[[60  2]
 [ 2 78]]

Rapport de classification :
              precision    recall  f1-score   support

           0      0.968     0.968     0.968        62
           1      0.975     0.975     0.975        80

    accuracy                          0.972       142
   macro avg      0

In [31]:
model = models_best_estimators.get("Modele_Régression_Logistique")
joblib.dump(model, '..\\models\\model.joblib')


['..\\models\\model.joblib']